In [ ]:
from backend.agent import workflow, AgentState
from uuid import uuid4
from langchain_core.messages import HumanMessage

result = None
config = {"configurable": {"thread_id": uuid4()}}
default = AgentState()
default.messages.append(
    HumanMessage(content="Can you create 5 posts from langchain subreddit? ")
)
while True:
    result = await workflow.ainvoke(input=result if result else default, config=config)
    next = input(result["messages"][-1].content)
    result["messages"].append(HumanMessage(content=next))

In [ ]:
import re

import requests
from bs4 import BeautifulSoup
from typing import Optional
def fetch_url_content(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve page with status code {response.status_code}")
        return None
    return response.content


def parse_article_content(article_url):
    content = fetch_url_content(article_url)
    if not content:
        return None

    article_soup = BeautifulSoup(content, "html.parser")
    paragraphs = article_soup.find_all("p")
    full_content = "\n".join([p.get_text() for p in paragraphs])

    unwanted_strings = [
        "Sign up\nSign in\nSign up\nSign in\nMariya Mansurova\nFollow\nTowards Data Science\n--\nListen\nShare",
        """\n--\n--\nTowards Data Science\nData & Product Analytics Lead at Wise | ClickHouse Evangelist\nHelp\nStatus\nAbout\nCareers\nPress\nBlog\nPrivacy\nTerms\nText to speech\nTeams""",
    ]
    for unwanted in unwanted_strings:
        full_content = full_content.replace(unwanted, "")

    full_content = re.sub(r"[^\x00-\x7F]+", "", full_content)
    return full_content


async def fetch_tds_articles():
    """
    Fetches the latest articles from Towards Data Science.
    
    Returns:
        ContentItem: A list of content items containing the latest articles.
    """
    screen_content = []
    page_content: Optional[str] = fetch_url_content(
        "https://towardsdatascience.com/latest"
    )
    if not page_content:
        raise Exception("Failed to fetch articles")

    soup: BeautifulSoup = BeautifulSoup(page_content, "html.parser")
    for article in soup.find_all("div", class_="postArticle", limit=5):
        link_tag = article.find("a", {"data-action": "open-post"})
        if not link_tag:
            continue

        full_content: Optional[str] = parse_article_content(link_tag["href"])
        screen_content.append(full_content)
    return screen_content

result = await fetch_tds_articles()
print(result)